# 02. Quét rank LoRA

Notebook này chạy thí nghiệm chính của phần deep learning: so sánh LoRA với các rank khác nhau. LoRA giữ nguyên trọng số gốc của model và chỉ học một cập nhật low-rank:

$$\Delta W = BA, \qquad A \in \mathbb{R}^{r \times d}, \quad B \in \mathbb{R}^{k \times r}.$$

Khi rank `r` nhỏ, số tham số cần huấn luyện giảm rất mạnh. Vì vậy notebook này đo trade-off giữa:

- Accuracy trên validation set.
- Số tham số trainable.
- Thời gian chạy.

Ba cấu hình được chạy là `lora_r2`, `lora_r4` và `lora_r8`.

## Bước 1. Đặt đúng thư mục làm việc

Cell này đưa notebook về thư mục gốc của project con `deep-learning`. Đây là bước nhỏ nhưng quan trọng, vì toàn bộ config và output đều dùng đường dẫn tương đối.

In [1]:
from pathlib import Path
import os

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
print(f"Thư mục làm việc: {Path.cwd()}")

Thư mục làm việc: d:\HK6\Machine learning\Lab_3\csc14005-introduction-to-machine-learning\lab-3\deep-learning


## Bước 2. Khai báo danh sách config LoRA

Mỗi file YAML cố định một cấu hình thí nghiệm. Điểm khác biệt chính là rank LoRA:

- `configs/lora_r2.yaml`: rank 2, ít tham số nhất.
- `configs/lora_r4.yaml`: rank 4, mức trung gian.
- `configs/lora_r8.yaml`: rank 8, nhiều tham số hơn, kỳ vọng có capacity cao hơn.

Trong code, `load_config` đọc YAML thành dictionary, còn `run_training` thực hiện toàn bộ pipeline: load tokenizer/model, inject LoRA, build dataloader, train, evaluate và lưu metrics.

In [2]:
from src.config import load_config
from src.train import run_training

lora_configs = [
    "configs/lora_r2.yaml",
    "configs/lora_r4.yaml",
    "configs/lora_r8.yaml",
]

d:\HK6\Machine learning\Lab_3\csc14005-introduction-to-machine-learning\lab-3\deep-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Bước 3. Chạy lần lượt từng rank

Cell này chạy từng config trong `lora_configs`. Sau mỗi lần train, notebook in ra kết quả tóm tắt để kiểm tra nhanh.

Các output quan trọng:

- `best_acc`: accuracy tốt nhất trên validation set.
- `trainable_params`: số tham số thực sự được cập nhật.

Trong lần chạy đã dùng cho báo cáo, kết quả là:

| Config | Trainable params | Best validation accuracy |
| --- | ---: | ---: |
| `lora_r2` | 2,306 | 0.7385 |
| `lora_r4` | 4,354 | 0.7351 |
| `lora_r8` | 8,450 | 0.7511 |

Điểm đáng chú ý là rank cao hơn không tự động tốt hơn trong setup nhỏ này. Rank 8 cho kết quả tốt nhất trong ba cấu hình LoRA.

In [3]:
lora_metrics = []
for config_path in lora_configs:
    print(f"\nĐang chạy {config_path}")
    metrics = run_training(load_config(config_path))
    lora_metrics.append(metrics)
    print(
        metrics["run_name"],
        "best_acc=", round(metrics["best_val_accuracy"], 4),
        "trainable_params=", metrics["model"]["trainable_params"],
    )


Đang chạy configs/lora_r2.yaml


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


lora_r2 best_acc= 0.7248 trainable_params= 2306

Đang chạy configs/lora_r4.yaml


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


lora_r4 best_acc= 0.7351 trainable_params= 4354

Đang chạy configs/lora_r8.yaml


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


lora_r8 best_acc= 0.7511 trainable_params= 8450


## Bước 4. Diễn giải kết quả

Sau khi chạy xong, mỗi cấu hình sẽ có một thư mục riêng trong `results/runs/`. Ví dụ:

- `results/runs/lora_r2/metrics.json`
- `results/runs/lora_r4/metrics.json`
- `results/runs/lora_r8/metrics.json`

Khi so với full fine-tuning trong notebook baseline, LoRA rank 2 chỉ dùng `2,306` tham số trainable thay vì `4,386,178`, tức ít hơn khoảng 1902 lần. Accuracy giảm từ `0.7661` xuống `0.7385`, mất khoảng 2.75 điểm phần trăm. Đây là minh họa trực tiếp cho thông điệp của tutorial: tensor/low-rank factorization giúp giảm rất mạnh chi phí cập nhật tham số, trong khi vẫn giữ được hiệu năng tương đối tốt.

Bước tiếp theo là chạy `04_visualize.ipynb` để sinh `metrics.csv` và các hình cho báo cáo.